# ArgLLM baseline (DF-QuAD) — DeepSeek-V3 on αNLI (500 claims, earlier sample from the training split)

This is the **earlier adaptation** of the ArgLLM prompts described in Section 5.1 of the paper: the model writes its own facts, and the argument prompt asks for one argument on each side instead of allowing the model to decline. Same 500 claims as `ours_deepseek_anli.ipynb`. Strengths are computed with the authors' DF-QuAD implementation (Uncertainpy). In `dfquad_pred`, the second field of the claim `Argument` is only its text label; DF-QuAD uses the base strength, never the gold label.

Run on Kaggle in batches of 100; the API key is read from Kaggle secrets (`DEEPSEEK_API_KEY`). The lines starting with `>>> Extension` print numbers copied from the extension notebook for comparison; they are not computed here.

In [ ]:
!pip install -q openai datasets scikit-learn
import os, sys, glob, re, time, pickle, random
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/anli_dfquad"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
MODEL_NAME = "deepseek-chat"

# SAME αNLI claims (random_state=42, identical to extension file)
ds = load_dataset("allenai/art", split="train", trust_remote_code=False)
adf = ds.to_pandas()
random.seed(42)
rows = []
sample = adf.sample(400, random_state=42).reset_index(drop=True)
for _, r in sample.iterrows():
    o1 = str(r["observation_1"]).strip(); o2 = str(r["observation_2"]).strip()
    correct = str(r["hypothesis_1"] if r["label"]==1 else r["hypothesis_2"]).strip()
    wrong   = str(r["hypothesis_2"] if r["label"]==1 else r["hypothesis_1"]).strip()
    rows.append({"claim": f"{o1} {o2} This is because: {correct}", "gold": 1})
    rows.append({"claim": f"{o1} {o2} This is because: {wrong}", "gold": 0})
cdf = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
t = cdf[cdf["gold"]==1].head(250); f = cdf[cdf["gold"]==0].head(250)
baf_df = pd.concat([t,f]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("αNLI claims:", len(baf_df), "| first:", baf_df.iloc[0]["claim"][:60])

def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks.
For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion. NOT about whether the claim is true or false.
Claim: "{claim}"
Return 3-5 atomic facts, one per line. Do not conclude."""
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=400)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            return ""
    return ""
print("✅ CELL 1 ready")

README.md: 0.00B [00:00, ?B/s]

anli/validation-00000-of-00001.parquet:   0%|          | 0.00/209k [00:00<?, ?B/s]

anli/train-00000-of-00001.parquet:   0%|          | 0.00/8.98M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1532 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/169654 [00:00<?, ? examples/s]

αNLI claims: 500 | first: My sister in law was a trolley driver. An inspector restarte
✅ CELL 1 ready


In [ ]:
if not glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True):
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
UNC = glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True)[0]
sys.path.insert(0, UNC)
from uncertainpy.gradual import BAG, Argument
from uncertainpy.gradual.algorithms.Acyclic import computeStrengthValues
from uncertainpy.gradual.semantics.modular import ProductAggregation, LinearInfluence
agg_f = ProductAggregation(); inf_f = LinearInfluence(conservativeness=1)

def llm_raw(prompt, mt=200):
    for _ in range(5):
        try:
            r=client.chat.completions.create(model=MODEL_NAME,messages=[{"role":"user","content":prompt}],temperature=0.7,max_tokens=mt,top_p=0.95)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            time.sleep(3)
    return ""
def am_prompt(claim, facts, support):
    verb="supporting" if support else "attacking"; rel="support" if support else "attack"
    return (f'Please provide a single short argument {verb} the following claim, using the facts as context. '
            f'Construct the argument so it refers to the truthfulness of the claim. '
            f'You must always produce a {rel} argument, even if weak; do not return N/A.\n'
            f'Facts:\n{facts}\n\nClaim: {claim}\nArgument:')
def ue_arg_prompt(argument, claim, support):
    favour="in favour of" if support else "against"; rel="supports" if support else "refutes"
    return (f'You are an analyst evaluating arguments. For the argument:\n\n"{argument}"\n\n'
            f'give your confidence it makes a compelling case {favour}:\n\n"{claim}"\n\n'
            f'Base it on how well the argument {rel} the statement and its accuracy. Respond 0-100%. Format:\nLikelihood:')
def ue_topic_prompt(claim, facts):
    return (f'You are an analyst. Using these facts as context:\n{facts}\n\nFor the statement:\n\n"{claim}"\n\n'
            f'give your confidence it is correct and truthful, 0-100%. Format:\nLikelihood:')
def gen_argument(claim, facts, support):
    for _ in range(2):
        txt=llm_raw(am_prompt(claim,facts,support),150); txt=re.sub(r'\*+','',txt); txt=re.sub(r'^\s*Argument\s*:\s*','',txt,flags=re.I).strip()
        lines=[l.strip() for l in txt.split("\n") if l.strip()]; txt=lines[0] if lines else txt
        if txt and "N/A" not in txt: return txt.strip('"').strip()
    return "N/A"
def get_lik(prompt):
    txt=llm_raw(prompt,150); m=re.search(r'Likelihood:\s*([0-9]{1,3})\s*%',txt) or re.search(r'([0-9]{1,3})\s*%',txt)
    return max(0,min(1,int(m.group(1))/100)) if m else 0.5
def arg_base(a,c,s): return None if a=="N/A" else get_lik(ue_arg_prompt(a,c,s))
def topic_base(c,f): return get_lik(ue_topic_prompt(c,f))
print("✅ CELL 2 ready — DF-QuAD solver loaded")

✅ CELL 2 ready — DF-QuAD solver loaded


In [ ]:
DFQ_CKPT = WORK_DIR / "anli_dfquad_ckpt.pkl"
def load_dfq(): return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d): pickle.dump(d, open(DFQ_CKPT,"wb"))

def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (DF-QuAD)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]; claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True)
            att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts,
                   "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True),
                   "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done")

run_dfquad_batch(100)

Have 0/500. Running 100 (DF-QuAD)...
  10/100 → 10 total
  20/100 → 20 total
  30/100 → 30 total
  40/100 → 40 total
  50/100 → 50 total
  60/100 → 60 total
  70/100 → 70 total
  80/100 → 80 total
  90/100 → 90 total
  100/100 → 100 total
✅ 100/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== αNLI DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension: ~69% | net_support: 54% <<<")

=== αNLI DF-QuAD (100) ===
  DF-QuAD 0.5 Base (fair): 62.0%  (F1 32.1)  TN=53 FP=1 FN=37 TP=9
  DF-QuAD Est. Base: 77.0%  (F1 74.7)  TN=43 FP=11 FN=12 TP=34

  >>> Extension: ~69% | net_support: 54% <<<


In [ ]:
run_dfquad_batch(100)

Have 100/500. Running 100 (DF-QuAD)...
  10/100 → 110 total
  20/100 → 120 total
  30/100 → 130 total
  40/100 → 140 total
  50/100 → 150 total
  60/100 → 160 total
  70/100 → 170 total
  80/100 → 180 total
  90/100 → 190 total
  100/100 → 200 total
✅ 200/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== αNLI DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension: 69.5% | net_support: 54% <<<")

=== αNLI DF-QuAD (200) ===
  DF-QuAD 0.5 Base (fair): 54.0%  (F1 25.8)  TN=92 FP=3 FN=89 TP=16
  DF-QuAD Est. Base: 74.0%  (F1 73.5)  TN=76 FP=19 FN=33 TP=72

  >>> Extension: 69.5% | net_support: 54% <<<


In [ ]:
run_dfquad_batch(100)

Have 200/500. Running 100 (DF-QuAD)...
  10/100 → 210 total
  20/100 → 220 total
  30/100 → 230 total
  40/100 → 240 total
  50/100 → 250 total
  60/100 → 260 total
  70/100 → 270 total
  80/100 → 280 total
  90/100 → 290 total
  100/100 → 300 total
✅ 300/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== αNLI DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension (500): 69.6% cred / 71.4% skep | net_support: 58.4% <<<")

=== αNLI DF-QuAD (300) ===
  DF-QuAD 0.5 Base (fair): 54.3%  (F1 23.5)  TN=142 FP=8 FN=129 TP=21
  DF-QuAD Est. Base: 77.7%  (F1 75.8)  TN=128 FP=22 FN=45 TP=105

  >>> Extension (500): 69.6% cred / 71.4% skep | net_support: 58.4% <<<


In [ ]:
run_dfquad_batch(100)

Have 300/500. Running 100 (DF-QuAD)...
  10/100 → 310 total
  20/100 → 320 total
  30/100 → 330 total
  40/100 → 340 total
  50/100 → 350 total
  60/100 → 360 total
  70/100 → 370 total
  80/100 → 380 total
  90/100 → 390 total
  100/100 → 400 total
✅ 400/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== αNLI DF-QuAD ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n  >>> Extension (500): 69.6% cred / 71.4% skep | net_support: 58.4% <<<")

=== αNLI DF-QuAD (400) ===
  DF-QuAD 0.5 Base (fair): 52.2%  (F1 22.0)  TN=182 FP=13 FN=178 TP=27
  DF-QuAD Est. Base: 76.5%  (F1 75.1)  TN=164 FP=31 FN=63 TP=142

  >>> Extension (500): 69.6% cred / 71.4% skep | net_support: 58.4% <<<


In [ ]:
run_dfquad_batch(100)

Have 400/500. Running 100 (DF-QuAD)...
  10/100 → 410 total
  20/100 → 420 total
  30/100 → 430 total
  40/100 → 440 total
  50/100 → 450 total
  60/100 → 460 total
  70/100 → 470 total
  80/100 → 480 total
  90/100 → 490 total
  100/100 → 500 total
✅ 500/500 done


In [ ]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n=len(idxs)
print(f"=== αNLI DF-QuAD FINAL ({n}) ===")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    print(f"  DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"\n=== FULL αNLI COMPARISON (500) ===")
print(f"  Extension skeptical:  71.4%")
print(f"  Extension credulous:  69.6%")
print(f"  net_support:          58.4%")
print(f"  DF-QuAD Est. Base:    (above)")
print(f"  DF-QuAD 0.5 Base:     (above)  ← FAIR comparison")

=== αNLI DF-QuAD FINAL (500) ===
  DF-QuAD 0.5 Base (fair): 52.6%  (F1 20.2)  TN=233 FP=17 FN=220 TP=30
  DF-QuAD Est. Base: 75.8%  (F1 73.6)  TN=210 FP=40 FN=81 TP=169

=== FULL αNLI COMPARISON (500) ===
  Extension skeptical:  71.4%
  Extension credulous:  69.6%
  net_support:          58.4%
  DF-QuAD Est. Base:    (above)
  DF-QuAD 0.5 Base:     (above)  ← FAIR comparison


In [ ]:
dfq = load_dfq()
results = [dfq[k] for k in sorted(dfq)]
n = len(results)
has_sup = sum(1 for r in results if r["sup"]!="N/A" and r["sup_base"] is not None)
has_att = sum(1 for r in results if r["att"]!="N/A" and r["att_base"] is not None)
sup_s = [r["sup_base"] for r in results if r["sup_base"] is not None]
att_s = [r["att_base"] for r in results if r["att_base"] is not None]
both = [(r["sup_base"],r["att_base"]) for r in results if r["sup_base"] is not None and r["att_base"] is not None]
att_wins = sum(1 for s,a in both if a>s)
print(f"=== αNLI DF-QuAD structure ({n}) ===")
print(f"Has supporter: {has_sup}/{n} | Has attacker: {has_att}/{n}")
print(f"Supporter strength: mean={np.mean(sup_s):.2f}")
print(f"Attacker strength:  mean={np.mean(att_s):.2f}")
print(f"Attacker STRONGER than supporter: {att_wins}/{len(both)} ({att_wins/len(both)*100:.1f}%)")
print(f"→ explains the FN collapse (F1 20.2, FN=220): attacker dominates from neutral base")

=== αNLI DF-QuAD structure (500) ===
Has supporter: 500/500 | Has attacker: 500/500
Supporter strength: mean=0.48
Attacker strength:  mean=0.76
Attacker STRONGER than supporter: 388/500 (77.6%)
→ explains the FN collapse (F1 20.2, FN=220): attacker dominates from neutral base


In [ ]:
dfq = load_dfq()
results = [dfq[k] for k in sorted(dfq)]

def dfquad_strength(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

def show_dfq(v, est):
    s = dfquad_strength(v, est)
    pred = int(s > 0.5)
    gold = "TRUE" if v["gold"]==1 else "FALSE"
    pr = "TRUE" if pred==1 else "FALSE"
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    print(f"CLAIM: {v['claim'][:95]}")
    print(f"  GOLD={gold} | PRED={pr} | strength={s:.3f} (τ={tau:.2f})")
    print(f"  SUP: [{v['sup_base'] if v['sup_base'] else 'N/A'}] {str(v['sup'])[:70]}")
    print(f"  ATT: [{v['att_base'] if v['att_base'] else 'N/A'}] {str(v['att'])[:70]}")
    print("-"*90)

def categorize(est):
    tp=tn=fp=fn=[]
    tp = [v for v in results if v["gold"]==1 and int(dfquad_strength(v,est)>0.5)==1]
    tn = [v for v in results if v["gold"]==0 and int(dfquad_strength(v,est)>0.5)==0]
    fp = [v for v in results if v["gold"]==0 and int(dfquad_strength(v,est)>0.5)==1]
    fn = [v for v in results if v["gold"]==1 and int(dfquad_strength(v,est)>0.5)==0]
    return tp,tn,fp,fn

for est, base_name in [(False, "0.5 BASE (fair)"), (True, "EST. BASE")]:
    tp,tn,fp,fn = categorize(est)
    print("\n" + "="*90)
    print(f"### {base_name} ###   (TP={len(tp)} TN={len(tn)} FP={len(fp)} FN={len(fn)})")
    print("="*90)
    print(f"\n--- TRUE POSITIVE (gold True, pred True) — showing 5 ---")
    for v in tp[:5]: show_dfq(v, est)
    print(f"\n--- TRUE NEGATIVE (gold False, pred False) — showing 5 ---")
    for v in tn[:5]: show_dfq(v, est)
    print(f"\n--- FALSE POSITIVE (gold False, pred True) — showing 5 ---")
    for v in fp[:5]: show_dfq(v, est)
    print(f"\n--- FALSE NEGATIVE (gold True, pred False) — showing 5 ---")
    for v in fn[:5]: show_dfq(v, est)


### 0.5 BASE (fair) ###   (TP=30 TN=233 FP=17 FN=220)

--- TRUE POSITIVE (gold True, pred True) — showing 5 ---
CLAIM: Becky's closet was very messy. Finally she felt better. This is because: Becky cleaned her clos
  GOLD=TRUE | PRED=TRUE | strength=0.550 (τ=0.50)
  SUP: [0.72] Becky felt better because she cleaned her closet, as cleaning is the a
  ATT: [0.62] The claim asserts that Becky felt better because she cleaned her close
------------------------------------------------------------------------------------------
CLAIM: Ann retired and was able to stay home all day and relax. She picked a local animal rescue to vo
  GOLD=TRUE | PRED=TRUE | strength=0.685 (τ=0.50)
  SUP: [0.72] The claim is truthful because the facts confirm that retirement enable
  ATT: [0.35] The claim says Ann's volunteer work was because she wanted to do somet
------------------------------------------------------------------------------------------
CLAIM: We shopped at our local bakery for years. My wife sa